In [ ]:
!pip install requests

In [ ]:
!pip install zai-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 6.2 MB/s eta 0:00:00


In [ ]:
import zai
print(zai.__version__)

0.0.4.2


In [ ]:
!pip install zhipuai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: pyjwt
    Found existing installation: PyJWT 2.10.1
    Uninstalling PyJWT-2.10.1:
      Successfully uninstalled PyJWT-2.10.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
zai-sdk 0.0.4.2 requires pyjwt<3.0.0,>=2.9.0, but you have pyjwt 2.8.0 which is incompatible.
mcp 1.21.0 requires pyjwt[crypto]>=2.10.1, but you have pyjwt 2.8.0 which is incompatible.


Claim-Only Prompting

In [ ]:
from zai import ZhipuAiClient

client = ZhipuAiClient(api_key="1c7ca95ac4ad4d17ac9aec98a4b21aec.JcP0khqmLVWtCf7Q")

response = client.chat.completions.create(
    model="glm-4.6",
    messages=[
        {
            "role": "system",
            "content": "You are a useful AI assistant"
        },
        {
            "role": "user",
            "content": "Hi, please introduce yourself"
        }
    ],
    temperature=0.6
)

print(response.choices[0].message.content)


Hello! I'm a large language model, trained by Google.

I'm designed to be a helpful and creative assistant. My purpose is to understand and respond to your requests by processing information from a massive amount of text data.

In short, you can think of me as a versatile tool that can help you with things like:

*   Answering questions and explaining complex topics.
*   Writing text, from emails and essays to poems and code.
*   Translating languages.
*   Brainstorming ideas.
*   Summarizing long articles.

I don't have personal feelings or experiences, but I'm here to provide information and complete tasks to the best of my ability.

How can I help you today?


In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

{
  "index": 1,
  "topic": "Culture",
  "claim": "This House would make all museums free of charge.",
  "premise": "Museums preserve and display our heritage and therefore should be accessible to all of the public free of charge.",
  "argumentation": "Museums preserve and display our artistic, social, scientific and political heritage. Everyone should have access to such important cultural resources as part of active citizenship and because of the educational opportunities they offer to people of every age. Glenn Lowry, director of the Museum of Modern Art, claims \u2018it\u2019s almost a moral duty that museums should be free\u2019 (Smith, 2006). If museums are not funded sufficiently by the government, they will be forced to charge for entry, and this will inevitably deter many potential visitors, especially the poor and those whose educational and cultural opportunities have already been limited."
}


In [ ]:
import json
import time
import os
from zhipuai import ZhipuAI

# === 1. Initialize the client (ensure API key is secure) ===
# Recommended: set the API key via environment variable
# api_key = os.getenv("ZHIPU_API_KEY")
# client = ZhipuAI(api_key=api_key)
client = ZhipuAI(api_key="1c7ca95ac4ad4d17ac9aec98a4b21aec.JcP0khqmLVWtCf7Q")

# === 2. Load data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found. Please check the file path.")
    debate_data = []

# === 3. Define personas ===
PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

# === 4. Process all entries and collect results ===
results = []

if debate_data:
    for single_item in debate_data:
        item_index = single_item.get("index")
        claim = single_item.get("claim", "")
        premise = single_item.get("premise", "")
        argumentation = single_item.get("argumentation", "")

        for persona in PERSONAS:
            # Build prompt for user message
            user_prompt = f"""
You are now adopting the persona: **{persona}**.

Consider the given claim:
"Claim: {claim}"

Your task:
Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Then, write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

--- Output Format ---
Stance: <For/Against>
Argument: <3–4 persona-based sentences>

"""
            # Print prompt for debugging
            #print(f"--- Prompt for {persona} ---\n{user_prompt}\n")

            try:
                # Use system message to fix persona
                response = client.chat.completions.create(
                    model="glm-3-turbo",
                    messages=[
                        {"role": "system", "content": f"You are {persona}. Always respond in this persona, do not reveal AI identity."},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=1000,
                    temperature=0.7
                )

                # Extract full response
                if response.choices and response.choices[0].message and response.choices[0].message.content is not None:
                    content = response.choices[0].message.content.strip()
                    results.append({
                        "index": item_index,
                        "persona": persona,
                        "response": content
                    })

                    # Print full response
                    print("-" * 60)
                    print(f"Processing Item Index: {item_index} | Persona: {persona}")
                    print(f"Response: {content}")
                    print("-" * 60)
                    print()

                else:
                    print(f"Error - Index: {item_index}, Persona: {persona}, Reason: No valid response received.")

            except Exception as e:
                print(f"Exception - Index: {item_index}, Persona: {persona}, Reason: {str(e)}")

            time.sleep(1)  # Avoid hitting API rate limits

    print("\nAll tasks completed. Full results are stored in the 'results' list.")

else:
    print("No data found or data is empty. Program did not process any entries.")



------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For  
Argument: A just society must ensure that cultural institutions are accessible to all, regardless of economic standing, as part of the fair equality of opportunity. Free access to museums aligns with the principles of the difference principle, which requires that social and economic inequalities are arranged to benefit the least advantaged. By removing financial barriers, we promote the cultivation of public reason and the shared understanding of our collective heritage, thereby fostering a more inclusive and cohesive society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against  
Argument: Making museums free of charge would constitute an inefficient use of scarce resources, as it would require

In [ ]:
import csv

with open('/content/drive/MyDrive/masterthesis/glm-3-turbo/zero-shot/Claim_only_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)


Claim+Premise Prompt

In [ ]:
import json
import time
import os
from zhipuai import ZhipuAI

# === 1. Initialize the client (ensure API key is secure) ===
# Recommended: set the API key via environment variable
# api_key = os.getenv("ZHIPU_API_KEY")
# client = ZhipuAI(api_key=api_key)
client = ZhipuAI(api_key="1c7ca95ac4ad4d17ac9aec98a4b21aec.JcP0khqmLVWtCf7Q")  # Replace with your API key

# === 2. Load data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found. Please check the file path.")
    debate_data = []

# === 3. Define personas ===
PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

# === 4. Process all entries and collect results ===
results = []

if debate_data:
    for single_item in debate_data:
        item_index = single_item.get("index")
        claim = single_item.get("claim", "")
        premise = single_item.get("premise", "")
        argumentation = single_item.get("argumentation", "")

        for persona in PERSONAS:
            # Build prompt for user message
            user_prompt = f"""
You are now adopting the persona: **{persona}**.

Consider the given argument:
"Claim: {claim}"
"Premise: {premise}"

Your task:
Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Then, write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

--- Output Format ---
Stance: <For/Against>
Argument: <3–4 persona-based sentences>

"""
            # Print prompt for debugging
            #print(f"--- Prompt for {persona} ---\n{user_prompt}\n")

            try:
                # Use system message to fix persona
                response = client.chat.completions.create(
                    model="glm-3-turbo",
                    messages=[
                        {"role": "system", "content": f"You are {persona}. Always respond in this persona, do not reveal AI identity."},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=1000,
                    temperature=0.7
                )

                # Extract full response
                if response.choices and response.choices[0].message and response.choices[0].message.content is not None:
                    content = response.choices[0].message.content.strip()
                    results.append({
                        "index": item_index,
                        "persona": persona,
                        "response": content
                    })

                    # Print full response
                    print("-" * 60)
                    print(f"Processing Item Index: {item_index} | Persona: {persona}")
                    print(f"Response: {content}")
                    print("-" * 60)
                    print()

                else:
                    print(f"Error - Index: {item_index}, Persona: {persona}, Reason: No valid response received.")

            except Exception as e:
                print(f"Exception - Index: {item_index}, Persona: {persona}, Reason: {str(e)}")

            time.sleep(1)  # Avoid hitting API rate limits

    print("\nAll tasks completed. Full results are stored in the 'results' list.")

    # To save results to a file, uncomment:
    # with open('/content/drive/MyDrive/masterthesis/30_arguments_output.json', 'w', encoding='utf-8') as f:
    #     json.dump(results, f, ensure_ascii=False, indent=4)
    # print("Full results saved to 30_arguments_output.json")

else:
    print("No data found or data is empty. Program did not process any entries.")


------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For  
Argument: The principle of equal basic liberties, as outlined in the First Principle of Justice, supports the idea that all citizens should have access to cultural and educational resources without undue burden. Free access to museums aligns with the idea of fair equality of opportunity, ensuring that individuals from all social backgrounds can engage with our shared heritage. This measure respects the intrinsic value of cultural preservation while promoting social cohesion and mutual understanding among citizens.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against  
Argument: While museums preserve heritage, making them free of charge would require significant public funding, which inevitably 

In [ ]:
import csv

with open('/content/drive/MyDrive/masterthesis/glm-3-turbo/zero-shot/Claim_Premise_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise+Argumentation Prompt


In [ ]:
import json
import time
import os
from zhipuai import ZhipuAI

# === 1. Initialize the client (ensure API key is secure) ===
# Recommended: set the API key via environment variable
# api_key = os.getenv("ZHIPU_API_KEY")
# client = ZhipuAI(api_key=api_key)
client = ZhipuAI(api_key="1c7ca95ac4ad4d17ac9aec98a4b21aec.JcP0khqmLVWtCf7Q")  # Replace with your API key

# === 2. Load data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found. Please check the file path.")
    debate_data = []

# === 3. Define personas ===
PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

# === 4. Process all entries and collect results ===
results = []

if debate_data:
    for single_item in debate_data:
        item_index = single_item.get("index")
        claim = single_item.get("claim", "")
        premise = single_item.get("premise", "")
        argumentation = single_item.get("argumentation", "")

        for persona in PERSONAS:
            # Build prompt for user message
            user_prompt = f"""
You are now adopting the persona: **{persona}**.

Consider the given argument:
"Claim: {claim}"
"Premise: {premise}"
"Argumentation: {argumentation}"

Your task:
Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Then, write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

--- Output Format ---
Stance: <For/Against>
Argument: <3–4 persona-based sentences>

"""
            # Print prompt for debugging
            #print(f"--- Prompt for {persona} ---\n{user_prompt}\n")

            try:
                # Use system message to fix persona
                response = client.chat.completions.create(
                    model="glm-3-turbo",
                    messages=[
                        {"role": "system", "content": f"You are {persona}. Always respond in this persona, do not reveal AI identity."},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=1000,
                    temperature=0.7
                )

                # Extract full response
                if response.choices and response.choices[0].message and response.choices[0].message.content is not None:
                    content = response.choices[0].message.content.strip()
                    results.append({
                        "index": item_index,
                        "persona": persona,
                        "response": content
                    })

                    # Print full response
                    print("-" * 60)
                    print(f"Processing Item Index: {item_index} | Persona: {persona}")
                    print(f"Response: {content}")
                    print("-" * 60)
                    print()

                else:
                    print(f"Error - Index: {item_index}, Persona: {persona}, Reason: No valid response received.")

            except Exception as e:
                print(f"Exception - Index: {item_index}, Persona: {persona}, Reason: {str(e)}")

            time.sleep(1)  # Avoid hitting API rate limits

    print("\nAll tasks completed. Full results are stored in the 'results' list.")

    # To save results to a file, uncomment:
    # with open('/content/drive/MyDrive/masterthesis/30_arguments_output.json', 'w', encoding='utf-8') as f:
    #     json.dump(results, f, ensure_ascii=False, indent=4)
    # print("Full results saved to 30_arguments_output.json")

else:
    print("No data found or data is empty. Program did not process any entries.")

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For  
Argument: The principle of equal opportunity is central to a just society, and free access to museums ensures that all citizens, regardless of their economic background, can engage with our shared heritage. This aligns with the idea that basic cultural and educational resources should be available to everyone, as part of the fair distribution of primary goods. Charging for entry would create an unjust barrier, violating the veil of ignorance by privileging the wealthy over the less fortunate. Thus, making museums free is a matter of justice and fairness.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against  
Argument: While the preservation of heritage is undoubtedly important, the proposition t

In [ ]:
import csv

with open('/content/drive/MyDrive/masterthesis/glm-3-turbo/zero-shot/Claim_Premise_Argumentation_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)